# WNBA Draft Fit Predictor — Validation

This notebook tracks how the 2026 rookie class is actually performing
compared to our predicted Rookie Impact Scores.

Updated as the 2026 WNBA season progresses.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load predictions
predictions = pd.read_csv("../data/processed/predictions_2026.csv")

print("Predicted RIS for 2026 class:")
print(predictions[['player', 'wnba_team', 'draft_pick', 'predicted_RIS']]
      .sort_values('predicted_RIS', ascending=False)
      .to_string())

Predicted RIS for 2026 class:
              player           wnba_team  draft_pick  predicted_RIS
7       Lauren Betts  Washington Mystics           4          0.595
8        Madina Okot       Atlanta Dream          13          0.521
6          Kiki Rice       Toronto Tempo           6          0.496
3   Flau'jae Johnson       Seattle Storm           8          0.486
4    Gabriela Jaquez         Chicago Sky           5          0.481
9       Olivia Miles      Minnesota Lynx           2          0.476
1          Azzi Fudd        Dallas Wings           1          0.414
10     Raven Johnson       Indiana Fever          10          0.333
11        Taina Mair       Seattle Storm          14          0.326
5   Gianna Kneepkens     Connecticut Sun          15          0.301
0     Angela Dugalic  Washington Mystics           9          0.300
2      Cotie McMahon  Washington Mystics          11          0.288


In [2]:
# Load current 2026 WNBA stats
current_stats = pd.read_csv("../data/external/wnba_2026_current_stats.csv")
print(current_stats.shape)
print(current_stats.columns.tolist())
print(current_stats.head())

(155, 28)
['Player', 'Team', 'Pos', 'G', 'MP', 'G.1', 'GS', 'MP.1', 'FG', 'FGA', 'FG%', '3P', '3PA', '3P%', '2P', '2PA', '2P%', 'FT', 'FTA', 'FT%', 'ORB', 'TRB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS']
             Player Team Pos  G  MP  G.1  GS  MP.1   FG   FGA  ...  FTA  \
0    Julie Allemand  TOR   G  1  30    1   1  30.0  1.0   2.0  ...  0.0   
1  Laeticia Amihere  GSV   F  2  40    2   0  20.0  3.0   5.0  ...  3.5   
2    Georgia Amoore  WAS   G  2  42    2   2  21.0  2.5   8.5  ...  0.0   
3    Pauline Astier  NYL   G  2  55    2   2  27.5  4.0   7.0  ...  5.0   
4      Ariel Atkins  LAS   G  1  32    1   1  32.0  3.0  12.0  ...  2.0   

     FT%  ORB  TRB  AST  STL  BLK  TOV   PF   PTS  
0    NaN  0.0  3.0  2.0  1.0  0.0  0.0  1.0   3.0  
1  0.571  2.0  5.0  3.0  0.0  1.5  1.5  4.0   8.0  
2    NaN  0.0  2.0  5.0  1.5  0.0  3.0  2.0   6.5  
3  0.700  1.0  3.5  5.5  1.5  0.0  1.0  4.5  11.5  
4  1.000  2.0  4.0  3.0  3.0  0.0  3.0  2.0   8.0  

[5 rows x 28 columns]


In [3]:
# Filter to just our 2026 rookies
rookie_names = predictions['player'].tolist()

# Current stats for our rookies
current_rookies = current_stats[current_stats['Player'].isin(rookie_names)].copy()

print(f"Rookies found in current stats: {len(current_rookies)}")
print(current_rookies[['Player', 'Team', 'G', 'MP.1', 'PTS', 'TRB', 'AST']].to_string())

Rookies found in current stats: 9
               Player Team  G  MP.1   PTS  TRB  AST
10       Lauren Betts  WAS  2  15.0   3.5  3.5  1.0
50          Azzi Fudd  DAL  1  18.0   3.0  1.0  0.0
77    Gabriela Jaquez  CHI  1  32.0  10.0  7.0  2.0
79   Flau'jae Johnson  SEA  2  27.5  14.0  4.0  1.5
80      Raven Johnson  IND  1  12.0   4.0  2.0  2.0
85   Gianna Kneepkens  CON  2  10.5   4.5  3.0  1.0
101      Olivia Miles  MIN  1  34.0  21.0  3.0  8.0
110       Madina Okot  ATL  1  10.0   8.0  4.0  0.0
119         Kiki Rice  TOR  1  18.0   0.0  3.0  1.0


In [4]:
# Merge predictions with current stats
current_rookies = current_rookies.rename(columns={'Player': 'player'})

validation_df = predictions[['player', 'wnba_team', 'draft_pick', 'predicted_RIS']].merge(
    current_rookies[['player', 'G', 'MP.1', 'PTS', 'TRB', 'AST']],
    on='player',
    how='left'
)

validation_df = validation_df.rename(columns={'MP.1': 'mpg', 'PTS': 'ppg', 
                                               'TRB': 'rpg', 'AST': 'apg'})

print(validation_df[['player', 'predicted_RIS', 'G', 'ppg', 'rpg', 'apg']]
      .sort_values('predicted_RIS', ascending=False)
      .to_string())

              player  predicted_RIS    G   ppg  rpg  apg
7       Lauren Betts          0.595  2.0   3.5  3.5  1.0
8        Madina Okot          0.521  1.0   8.0  4.0  0.0
6          Kiki Rice          0.496  1.0   0.0  3.0  1.0
3   Flau'jae Johnson          0.486  2.0  14.0  4.0  1.5
4    Gabriela Jaquez          0.481  1.0  10.0  7.0  2.0
9       Olivia Miles          0.476  1.0  21.0  3.0  8.0
1          Azzi Fudd          0.414  1.0   3.0  1.0  0.0
10     Raven Johnson          0.333  1.0   4.0  2.0  2.0
11        Taina Mair          0.326  NaN   NaN  NaN  NaN
5   Gianna Kneepkens          0.301  2.0   4.5  3.0  1.0
0     Angela Dugalic          0.300  NaN   NaN  NaN  NaN
2      Cotie McMahon          0.288  NaN   NaN  NaN  NaN


In [5]:
# Save validation snapshot
validation_df['snapshot_date'] = '2026-05-12'
validation_df['games_into_season'] = validation_df['G'].fillna(0).astype(int)

validation_df.to_csv("../data/processed/validation_snapshot.csv", index=False)
print("Saved validation_snapshot.csv!")

print("\nEarly season notes:")
print("- Only 1-2 games played per player — far too early to validate")
print("- Olivia Miles (21 PPG) and Flau'jae Johnson (14 PPG) strong early")
print("- Lauren Betts in limited minutes so far")
print("- Cotie McMahon, Taina Mair, Angela Dugalic not yet in stats")

Saved validation_snapshot.csv!

Early season notes:
- Only 1-2 games played per player — far too early to validate
- Olivia Miles (21 PPG) and Flau'jae Johnson (14 PPG) strong early
- Lauren Betts in limited minutes so far
- Cotie McMahon, Taina Mair, Angela Dugalic not yet in stats
